# Herausforderung: Analyse eines Textes über Data Science

In diesem Beispiel machen wir eine einfache Übung, die alle Schritte eines traditionellen Data-Science-Prozesses abdeckt. Du musst keinen Code schreiben, du kannst einfach auf die Zellen unten klicken, um sie auszuführen und das Ergebnis zu beobachten. Als Herausforderung wirst du ermutigt, diesen Code mit verschiedenen Daten auszuprobieren. 

## Ziel

In dieser Lektion haben wir verschiedene Konzepte im Zusammenhang mit Data Science besprochen. Versuchen wir, weitere verwandte Konzepte durch **Text Mining** zu entdecken. Wir beginnen mit einem Text über Data Science, extrahieren Schlüsselwörter daraus und versuchen dann, das Ergebnis zu visualisieren.

Als Text werde ich die Seite über Data Science von Wikipedia verwenden:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## Schritt 1: Die Daten beschaffen

Der erste Schritt in jedem Datenwissenschaftsprozess ist das Beschaffen der Daten. Wir werden die `requests` Bibliothek dafür verwenden:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## Schritt 2: Die Daten transformieren

Der nächste Schritt besteht darin, die Daten in eine für die Verarbeitung geeignete Form zu bringen. In unserem Fall haben wir den HTML-Quellcode der Seite heruntergeladen und müssen ihn in reinen Text umwandeln.

Es gibt viele Möglichkeiten, dies zu tun. Wir verwenden [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), eine beliebte Python-Bibliothek zum Parsen von HTML. BeautifulSoup ermöglicht es uns, gezielt bestimmte HTML-Elemente anzusteuern, sodass wir uns auf den Hauptartikelinhalt von Wikipedia konzentrieren und einige Navigationsmenüs, Seitenleisten, Footer und andere irrelevante Inhalte reduzieren können (obwohl einige Boilerplate-Texte möglicherweise weiterhin vorhanden sind).


Zuerst müssen wir die BeautifulSoup-Bibliothek für die HTML-Analyse installieren:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## Schritt 3: Erkenntnisse gewinnen

Der wichtigste Schritt ist, unsere Daten in eine Form zu bringen, aus der wir Erkenntnisse gewinnen können. In unserem Fall möchten wir Schlüsselwörter aus dem Text extrahieren und sehen, welche Schlüsselwörter bedeutungsvoller sind.

Wir werden die Python-Bibliothek [RAKE](https://github.com/aneesha/RAKE) zur Schlüsselwortextraktion verwenden. Zuerst installieren wir diese Bibliothek, falls sie nicht vorhanden ist: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

Die Hauptfunktionalität ist über das `Rake`-Objekt verfügbar, das wir mit einigen Parametern anpassen können. In unserem Fall setzen wir die Mindestlänge eines Schlüsselworts auf 5 Zeichen, die Mindesthäufigkeit eines Schlüsselworts im Dokument auf 3 und die maximale Anzahl von Wörtern in einem Schlüsselwort auf 2. Probieren Sie gerne andere Werte aus und beobachten Sie das Ergebnis.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Wir haben eine Liste von Begriffen zusammen mit dem zugehörigen Wichtigkeitsgrad erhalten. Wie Sie sehen können, sind die relevantesten Disziplinen, wie maschinelles Lernen und Big Data, ganz oben in der Liste vorhanden.

## Schritt 4: Visualisierung des Ergebnisses

Menschen können Daten am besten in visueller Form interpretieren. Daher ist es oft sinnvoll, die Daten zu visualisieren, um Erkenntnisse zu gewinnen. Wir können die `matplotlib`-Bibliothek in Python verwenden, um eine einfache Verteilung der Schlüsselwörter mit ihrer Relevanz darzustellen:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Es gibt jedoch eine noch bessere Möglichkeit, Wortfrequenzen zu visualisieren – mit einer **Wortwolke**. Wir müssen eine weitere Bibliothek installieren, um die Wortwolke aus unserer Schlüsselwortliste zu erstellen.


In [ ]:
!{sys.executable} -m pip install wordcloud

Das `WordCloud`-Objekt ist dafür verantwortlich, entweder ursprünglichen Text oder eine vorab berechnete Liste von Wörtern mit ihren Häufigkeiten zu übernehmen und ein Bild zurückzugeben, das dann mit `matplotlib` angezeigt werden kann:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

Wir können auch den Originaltext an `WordCloud` übergeben – schauen wir mal, ob wir ein ähnliches Ergebnis erhalten:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Sie können sehen, dass die Wortwolke jetzt beeindruckender aussieht, aber sie enthält auch viel Lärm (z. B. unzusammenhängende Wörter wie `Retrieved on`). Außerdem erhalten wir weniger Schlüsselwörter, die aus zwei Wörtern bestehen, wie *data scientist* oder *computer science*. Das liegt daran, dass der RAKE-Algorithmus bei der Auswahl guter Schlüsselwörter aus Text viel besser arbeitet. Dieses Beispiel verdeutlicht die Bedeutung der Datenvorverarbeitung und -bereinigung, da uns ein klares Bild am Ende ermöglicht, bessere Entscheidungen zu treffen.

In dieser Übung haben wir einen einfachen Prozess durchlaufen, um aus Wikipedia-Texten in Form von Schlüsselwörtern und einer Wortwolke etwas Bedeutung zu extrahieren. Dieses Beispiel ist recht einfach, zeigt aber gut alle typischen Schritte, die ein Data Scientist bei der Arbeit mit Daten durchführt – von der Datenbeschaffung bis zur Visualisierung.

In unserem Kurs werden wir all diese Schritte ausführlich besprechen. 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
